# 02 Autograd 和手写线性回归

本 notebook 对应学习计划的第 2 阶段。目标是不用 `nn.Linear` 和 optimizer，直接用 Tensor、计算图、`backward()` 和手动参数更新理解最小训练过程。

## 本节目标

- 理解计算图如何从 loss 连接到参数。
- 解释 `requires_grad`、`grad`、`backward()` 的职责。
- 说明为什么每轮训练前后要清零梯度。
- 区分 `torch.no_grad()`、`detach()`、`requires_grad=False`。
- 识别不合适 inplace 操作对 Autograd 的影响。
- 手写一个完整线性回归训练循环。

In [ ]:
from pathlib import Path
import sys

import torch

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from learn_pytorch import (
    get_device,
    make_linear_regression_data,
    mean_squared_error,
    predict_linear,
    train_manual_linear_regression,
)

torch.manual_seed(42)
device = get_device()

print(f"torch version: {torch.__version__}")
print(f"selected device: {device}")

In [ ]:
def describe_autograd(name: str, tensor: torch.Tensor) -> None:
    print(
        f"{name}: "
        f"shape={tuple(tensor.shape)}, "
        f"requires_grad={tensor.requires_grad}, "
        f"is_leaf={tensor.is_leaf}, "
        f"grad_fn={type(tensor.grad_fn).__name__ if tensor.grad_fn else None}, "
        f"grad={tensor.grad}"
    )

## 1. 一个最小计算图

从标量开始最容易看清楚链式法则。下面的函数是：

```text
y = x^2 + 3x
dy/dx = 2x + 3
x = 2 时，dy/dx = 7
```

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x

describe_autograd("x before backward", x)
describe_autograd("y before backward", y)

y.backward()

describe_autograd("x after backward", x)
print("analytical gradient:", 2 * 2.0 + 3)

assert torch.isclose(x.grad, torch.tensor(7.0))

第一性原理：`backward()` 从标量输出出发，沿计算图反向应用链式法则。`x` 是叶子 Tensor，所以默认保留 `.grad`；`y` 是中间结果，不是叶子 Tensor。

## 2. 叶子 Tensor 和中间 Tensor

手动创建且 `requires_grad=True` 的参数通常是叶子 Tensor。由运算产生的 Tensor 是中间 Tensor，默认不保留 `.grad`。如果确实想看中间 Tensor 的梯度，可以调用 `retain_grad()`。

In [ ]:
w = torch.tensor(1.5, requires_grad=True)
intermediate = w * 2
intermediate.retain_grad()
loss = intermediate ** 2

describe_autograd("w before backward", w)
describe_autograd("intermediate before backward", intermediate)

loss.backward()

describe_autograd("w after backward", w)
describe_autograd("intermediate after backward", intermediate)

## 3. 手写线性回归：一次 forward/backward/update

生成一组一维数据：

```text
y = 2x - 1 + noise
```

先只做一次训练步骤，观察 loss 和梯度。

In [ ]:
data = make_linear_regression_data(
    num_samples=8,
    weight=2.0,
    bias=-1.0,
    noise_std=0.0,
    seed=42,
    device=device,
)

weight = torch.tensor(0.0, device=device, requires_grad=True)
bias = torch.tensor(0.0, device=device, requires_grad=True)

prediction = predict_linear(data.x, weight, bias)
loss = mean_squared_error(prediction, data.y)

print("x shape:", data.x.shape)
print("y shape:", data.y.shape)
print("prediction shape:", prediction.shape)
print("loss:", float(loss.detach().cpu()))

loss.backward()

print("weight grad:", float(weight.grad.detach().cpu()))
print("bias grad:", float(bias.grad.detach().cpu()))

In [ ]:
learning_rate = 0.1

with torch.no_grad():
    weight -= learning_rate * weight.grad
    bias -= learning_rate * bias.grad
    weight.grad.zero_()
    bias.grad.zero_()

print("updated weight:", float(weight.detach().cpu()))
print("updated bias:", float(bias.detach().cpu()))
print("cleared weight grad:", weight.grad)
print("cleared bias grad:", bias.grad)

参数更新要放在 `torch.no_grad()` 里，因为更新参数本身不是模型 forward 的一部分，不应该继续构建新的计算图。

## 4. 完整训练循环

下面仍然不使用 `nn.Linear` 和 optimizer，只用原始 Tensor、Autograd 和手动 SGD。

In [ ]:
train_data = make_linear_regression_data(
    num_samples=128,
    weight=2.0,
    bias=-1.0,
    noise_std=0.05,
    seed=7,
    device=device,
)

result = train_manual_linear_regression(
    train_data.x,
    train_data.y,
    learning_rate=0.1,
    epochs=200,
    seed=0,
)

print("true weight:", train_data.true_weight)
print("learned weight:", result.weight)
print("true bias:", train_data.true_bias)
print("learned bias:", result.bias)
print("first loss:", result.losses[0])
print("last loss:", result.losses[-1])

assert result.losses[-1] < result.losses[0]
assert abs(result.weight - train_data.true_weight) < 0.15
assert abs(result.bias - train_data.true_bias) < 0.15

## 5. 梯度会累积：忘记清零会发生什么

PyTorch 默认累积梯度。这是为了支持梯度累积训练，但初学时最容易忘记清零。

In [ ]:
accum_weight = torch.tensor(0.0, device=device, requires_grad=True)
accum_bias = torch.tensor(0.0, device=device, requires_grad=True)

for step in range(3):
    pred = predict_linear(data.x, accum_weight, accum_bias)
    step_loss = mean_squared_error(pred, data.y)
    step_loss.backward()
    print(
        f"step={step}, "
        f"loss={float(step_loss.detach().cpu()):.4f}, "
        f"accumulated w.grad={float(accum_weight.grad.detach().cpu()):.4f}, "
        f"accumulated b.grad={float(accum_bias.grad.detach().cpu()):.4f}"
    )

# Clean up so later cells do not reuse accumulated gradients by accident.
accum_weight.grad.zero_()
accum_bias.grad.zero_()

如果每一轮代表一次独立更新，就要在更新后清零。否则 `.grad` 会混入历史梯度。

## 6. `no_grad()`、`detach()`、`requires_grad=False`

这三个概念常被混用，但语义不同。

In [ ]:
base = torch.tensor(2.0, requires_grad=True)
tracked = base * 3

with torch.no_grad():
    not_tracked = base * 3

detached = tracked.detach()
frozen = torch.tensor(2.0, requires_grad=False)
frozen_result = frozen * 3

describe_autograd("tracked", tracked)
describe_autograd("not_tracked", not_tracked)
describe_autograd("detached", detached)
describe_autograd("frozen_result", frozen_result)

print("detach shares storage with tracked:", detached.data_ptr() == tracked.data_ptr())

- `no_grad()`：上下文内不建图，常用于评估和参数更新。
- `detach()`：从已有图上切断当前 Tensor，通常仍共享底层数据。
- `requires_grad=False`：这个 Tensor 不参与梯度追踪，常用于冻结参数。

## 7. Inplace 操作的坑

带下划线的方法通常是 inplace 操作。不是所有 inplace 操作都错，但如果修改了 Autograd 反向传播需要的值，就会报错或导致错误结果。

In [ ]:
leaf = torch.ones(3, requires_grad=True)

try:
    leaf.add_(1.0)
except RuntimeError as error:
    print("inplace on leaf failed:", error)

safe_leaf = torch.ones(3, requires_grad=True)
with torch.no_grad():
    safe_leaf.add_(1.0)
print("safe update under no_grad:", safe_leaf)

In [ ]:
source = torch.ones(3, requires_grad=True)
middle = source * 2
output = (middle ** 2).sum()

try:
    middle.add_(1.0)
    output.backward()
except RuntimeError as error:
    print("inplace changed an intermediate needed by backward:", error)

学习阶段的保守策略：参数更新写在 `torch.no_grad()` 里；不确定是否安全时，先写非 inplace 版本。

## 8. 本节复盘

最小训练循环的本质：

```text
参数需要 requires_grad=True
loss 必须由参数参与计算得到
loss.backward() 把梯度写入叶子参数的 .grad
参数更新不应被记录进计算图
每轮更新后清空 .grad
```

In [ ]:
# Quick self-check: these assertions summarize the core autograd rules in this notebook.
check_x = torch.tensor(2.0, requires_grad=True)
check_y = check_x ** 2 + 3 * check_x
check_y.backward()

assert torch.isclose(check_x.grad, torch.tensor(7.0))
assert result.losses[-1] < result.losses[0]
assert abs(result.weight - train_data.true_weight) < 0.15
assert abs(result.bias - train_data.true_bias) < 0.15
print("stage 2 self-check passed")